In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

# 1. Carregar dados de treinamento com features extraídas
print("Carregando dados de treinamento...")
df_train = pd.read_csv('dataset/training_with_satellite_features.csv')

# Exibir informações sobre o dataset
print(f"Conjunto de dados carregado com {df_train.shape[0]} amostras e {df_train.shape[1]} colunas")
print("\nEstatísticas descritivas do UHI Index:")
print(df_train['UHI Index'].describe())

# 2. Tratamento de valores ausentes - preenchimento com a mediana
print("\nTratando valores ausentes...")
for col in df_train.columns:
    if df_train[col].dtype != 'object' and col != 'id':
        missing_count = df_train[col].isnull().sum()
        if missing_count > 0:
            print(f"Preenchendo {missing_count} valores nulos em '{col}' com mediana")
            df_train[col] = df_train[col].fillna(df_train[col].median())

# 3. Seleção de features, excluindo coordenadas geográficas e identificadores
X_train = df_train.drop(['id', 'UHI Index', 'Latitude', 'Longitude'], axis=1, errors='ignore')
y_train = df_train['UHI Index']

print("\nFeatures utilizadas no modelo:")
print(X_train.columns.tolist())

# 4. Exploração das correlações entre variáveis preditoras e alvo
plt.figure(figsize=(12, 10))
feature_corr = df_train[X_train.columns.tolist() + ['UHI Index']].corr()['UHI Index'].sort_values(ascending=False)
sns.heatmap(feature_corr.to_frame(), annot=True, cmap='coolwarm', fmt='.3f')
plt.title('Correlação entre features e índice UHI')
plt.tight_layout()
plt.savefig('feature_correlation.png')
plt.close()

print("\nCorrelações entre features e UHI Index:")
print(feature_corr)

# 5. Pré-processamento - opcionalmente escalar as features
print("\n=== Pré-processamento dos Dados ===")
# Criar uma cópia dos dados originais para referência
X_train_original = X_train.copy()

# Aplicar padronização às features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)

print("Estatísticas das features antes da padronização (primeiras 3 features):")
print(X_train[X_train.columns[:3]].describe().round(3))
print("\nEstatísticas das features após padronização (primeiras 3 features):")
print(X_train_scaled[X_train_scaled.columns[:3]].describe().round(3))

# 6. Implementação e comparação de modelos
print("\n=== Treinamento e Avaliação de Modelos ===")

# Definir modelos a serem comparados
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

# Ajuste fino de hiperparâmetros para o XGBoost
print("\nRealizando ajuste de hiperparâmetros para XGBoost...")
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}

xgb_model = xgb.XGBRegressor(random_state=42)
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"Melhor RMSE: {np.sqrt(-grid_search.best_score_):.6f}")

# Adicionar o modelo XGBoost otimizado
models['XGBoost Otimizado'] = grid_search.best_estimator_

# Configuração para validação cruzada
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Estrutura para armazenar todos os resultados
all_results = []

# Avaliar cada modelo com dados originais e padronizados
datasets = {
    'Dados Originais': X_train,
    'Dados Padronizados': X_train_scaled
}

for dataset_name, dataset in datasets.items():
    print(f"\n==== Avaliação com {dataset_name} ====")
    
    # Reinicializar resultados para este conjunto de dados
    cv_results = {}
    
    for name, model in models.items():
        print(f"\nAvaliando modelo: {name}")
        
        # Validação cruzada
        cv_scores = cross_val_score(model, dataset, y_train, cv=kfold, scoring='neg_mean_squared_error')
        rmse_scores = np.sqrt(-cv_scores)
        r2_scores = cross_val_score(model, dataset, y_train, cv=kfold, scoring='r2')
        
        # Armazenar resultados (usando dicionário simples)
        result_entry = {
            'Model': name,
            'Dataset': dataset_name,
            'RMSE': rmse_scores.mean(),
            'RMSE_std': rmse_scores.std(),
            'R2': r2_scores.mean(),
            'R2_std': r2_scores.std()
        }
        
        # Adicionar à lista de todos os resultados
        all_results.append(result_entry)
        
        # Armazenar no dicionário de resultados para este dataset
        cv_results[name] = result_entry
        
        print(f"  RMSE médio (validação cruzada): {rmse_scores.mean():.6f} ± {rmse_scores.std():.6f}")
        print(f"  R² médio (validação cruzada): {r2_scores.mean():.4f} ± {r2_scores.std():.4f}")
        
        # Treinar o modelo na amostra completa
        model.fit(dataset, y_train)
        
        # Avaliar importância das features
        if hasattr(model, 'feature_importances_'):
            feature_importance = pd.DataFrame({
                'Feature': X_train.columns,
                'Importance': model.feature_importances_
            }).sort_values('Importance', ascending=False)
            
            print("\nImportância das features:")
            print(feature_importance.head(10))  # Mostrar as 10 features mais importantes
            
            # Visualizar importância das features
            plt.figure(figsize=(10, 6))
            sns.barplot(x='Importance', y='Feature', data=feature_importance.head(10))
            plt.title(f'Top 10 Features Mais Importantes - {name} ({dataset_name})')
            plt.tight_layout()
            plt.savefig(f'{name.replace(" ", "_")}_{dataset_name.replace(" ", "_")}_feature_importance.png')
            plt.close()
        else:
            print("O modelo não suporta visualização de importância de features")
            
    # Comparar resultados dos modelos para este conjunto de dados
    results_df = pd.DataFrame.from_dict(cv_results, orient='index')
    print(f"\nComparação dos Modelos ({dataset_name}):")
    print(results_df[['RMSE', 'RMSE_std', 'R2', 'R2_std']])
    
    # Visualizar comparação dos modelos
    plt.figure(figsize=(12, 6))
    plt.bar(results_df.index, results_df['RMSE'], yerr=results_df['RMSE_std'], capsize=4, alpha=0.7)
    plt.title(f'Comparação de RMSE entre Modelos ({dataset_name})')
    plt.ylabel('RMSE (menor é melhor)')
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'model_comparison_rmse_{dataset_name.replace(" ", "_")}.png')
    plt.close()
    
    plt.figure(figsize=(12, 6))
    plt.bar(results_df.index, results_df['R2'], yerr=results_df['R2_std'], capsize=4, alpha=0.7)
    plt.title(f'Comparação de R² entre Modelos ({dataset_name})')
    plt.ylabel('R² (maior é melhor)')
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'model_comparison_r2_{dataset_name.replace(" ", "_")}.png')
    plt.close()

# Converter a lista de resultados para DataFrame para facilitar a análise
all_results_df = pd.DataFrame(all_results)
print("\nTodos os resultados combinados:")
print(all_results_df)

# 7. Selecionar o melhor modelo com base no RMSE
# Encontrar o índice do melhor modelo com base no RMSE mínimo
best_idx = all_results_df['RMSE'].idxmin()
best_result = all_results_df.iloc[best_idx]

best_model_name = best_result['Model']
best_dataset = best_result['Dataset']
rmse_value = best_result['RMSE']
r2_value = best_result['R2']

print(f"\nMelhor modelo baseado em RMSE: {best_model_name} treinado com {best_dataset}")
print(f"RMSE: {rmse_value:.6f}, R²: {r2_value:.4f}")

# Treinar o melhor modelo nos dados completos
best_model = models[best_model_name]
if best_dataset == 'Dados Padronizados':
    print("Usando dados padronizados para o modelo final")
    final_training_data = X_train_scaled
    # Salvar o scaler para uso posterior nas predições
    use_scaling = True
else:
    print("Usando dados originais para o modelo final")
    final_training_data = X_train
    use_scaling = False

best_model.fit(final_training_data, y_train)

# 8. Carregar features extraídas para os pontos de submissão
print("\nCarregando features para os pontos de submissão...")
submission_features = pd.read_csv('dataset/submission_features.csv')

# 9. Verificação de consistência entre features de treinamento e submissão
missing_cols = set(X_train.columns) - set(submission_features.columns)
extra_cols = set(submission_features.columns) - set(X_train.columns) - set(['Latitude', 'Longitude'])

if missing_cols:
    print(f"Atenção: Colunas ausentes nas features de submissão: {missing_cols}")
    # Adicionar colunas ausentes com valores nulos
    for col in missing_cols:
        submission_features[col] = np.nan

# 10. Tratamento de valores ausentes nas features de submissão
for col in X_train.columns:
    if col in submission_features.columns and submission_features[col].isnull().any():
        median_val = df_train[col].median()
        print(f"Preenchendo {submission_features[col].isnull().sum()} valores nulos em '{col}' com mediana: {median_val}")
        submission_features[col] = submission_features[col].fillna(median_val)

# 11. Seleção das mesmas features usadas no treinamento
X_submission = submission_features[X_train.columns]

# Aplicar a mesma transformação de escala se necessário
if use_scaling:
    X_submission = pd.DataFrame(scaler.transform(X_submission), columns=X_submission.columns)
    print("Aplicando padronização aos dados de submissão")

# 12. Geração de predições com o melhor modelo
print(f"\nGerando predições com o modelo {best_model_name}...")
predictions = best_model.predict(X_submission)

# 13. Integração das predições ao template de submissão
submission_df = pd.read_csv('dataset/submission_template.csv')
submission_df['UHI Index'] = predictions

# Visualização das predições
print("\nPrevisões para os pontos de submissão:")
print(submission_df.head())

# Estatísticas básicas das predições
print("\nEstatísticas das predições:")
print(submission_df['UHI Index'].describe())

# Visualizar distribuição das previsões
plt.figure(figsize=(10, 6))
sns.histplot(submission_df['UHI Index'], kde=True)
plt.title('Distribuição das Predições do Índice UHI')
plt.grid(True)
plt.savefig('predictions_distribution.png')
plt.close()

# 14. Salvar resultados
# Salvar arquivo de submissão com predições
model_identifier = f"{best_model_name.replace(' ', '_')}_{best_dataset.replace(' ', '_')}"
submission_file = f'submission_{model_identifier}.csv'
submission_df.to_csv(submission_file, index=False)
print(f"\nArquivo de submissão com predições salvo como '{submission_file}'")

# Comparar com a versão anterior do Random Forest
try:
    previous_submission = pd.read_csv('submission_final.csv')
    print("\nComparação com a submissão anterior:")
    print(f"  Média anterior: {previous_submission['UHI Index'].mean():.6f}")
    print(f"  Média nova: {submission_df['UHI Index'].mean():.6f}")
    print(f"  Diferença absoluta média: {np.abs(previous_submission['UHI Index'] - submission_df['UHI Index']).mean():.6f}")
    
    # Calcular correlação entre predições anteriores e novas
    correlation = np.corrcoef(previous_submission['UHI Index'], submission_df['UHI Index'])[0,1]
    print(f"  Correlação entre predições anteriores e novas: {correlation:.4f}")
    
    # Visualizar comparação
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Distribuição de densidade
    sns.kdeplot(previous_submission['UHI Index'], label='Modelo Anterior (RF)', ax=ax1)
    sns.kdeplot(submission_df['UHI Index'], label=f'Novo Modelo ({best_model_name})', ax=ax1)
    ax1.set_title('Comparação da Distribuição de Predições')
    ax1.legend()
    ax1.grid(True)
    
    # Scatter plot
    ax2.scatter(previous_submission['UHI Index'], submission_df['UHI Index'], alpha=0.5)
    ax2.plot([0.96, 1.04], [0.96, 1.04], 'r--', label='Linha 1:1')
    ax2.set_xlabel('Predições do Modelo Anterior')
    ax2.set_ylabel('Predições do Novo Modelo')
    ax2.set_title(f'Correlação: {correlation:.4f}')
    ax2.grid(True)
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig('predictions_comparison.png')
    plt.close()
    
    # Criar mapa de calor para visualizar a diferença espacial entre as predições
    plt.figure(figsize=(12, 10))
    diff = submission_df['UHI Index'] - previous_submission['UHI Index']
    sc = plt.scatter(submission_df['Longitude'], submission_df['Latitude'], 
                   c=diff, cmap='coolwarm', alpha=0.8, s=30,
                   vmin=-max(abs(diff)), vmax=max(abs(diff)))
    plt.colorbar(sc, label='Diferença nas Predições (Novo - Antigo)')
    plt.title('Diferenças Espaciais nas Predições')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.grid(True, alpha=0.3)
    plt.savefig('spatial_prediction_differences.png')
    plt.close()
except Exception as e:
    print(f"Erro ao comparar com submissão anterior: {e}")

# Salvar um relatório de desempenho dos modelos
all_results_df.to_csv('model_performance_comparison.csv')
print("Comparação de desempenho dos modelos salva em 'model_performance_comparison.csv'")

print("\nProcesso de modelagem e predição concluído com sucesso!")

Carregando dados de treinamento...
Conjunto de dados carregado com 11229 amostras e 20 colunas

Estatísticas descritivas do UHI Index:
count    11229.000000
mean         1.000001
std          0.016238
min          0.956122
25%          0.988577
50%          1.000237
75%          1.011176
max          1.046036
Name: UHI Index, dtype: float64

Tratando valores ausentes...

Features utilizadas no modelo:
['S2_B01', 'S2_B02', 'S2_B03', 'S2_B04', 'S2_B05', 'S2_B06', 'S2_B07', 'S2_B08', 'S2_B8A', 'S2_B11', 'S2_B12', 'NDVI_S2', 'NDBI_S2', 'NDWI_S2', 'NDVI_LS', 'LST_LS']

Correlações entre features e UHI Index:
UHI Index    1.000000
LST_LS       0.419292
S2_B01       0.333458
NDWI_S2      0.236544
NDBI_S2      0.211681
S2_B12       0.185032
S2_B05       0.166922
S2_B04       0.158302
S2_B02       0.154430
S2_B03       0.149865
S2_B11       0.127232
S2_B06      -0.025881
S2_B07      -0.087323
S2_B8A      -0.102365
S2_B08      -0.115259
NDVI_S2     -0.231786
NDVI_LS     -0.269039
Name: UHI Index